In [1]:
import polars as pl
df1=pl.read_parquet("residual_load_results.parquet123")
df2=pl.read_parquet("residual_load_results.parquet4")
df3=pl.read_parquet("residual_load_results.parquet5")
df4=pl.read_parquet("residual_load_results.parquet6")
df5=pl.read_parquet("residual_load_results.parquet7")
df6=pl.read_parquet("residual_load_results.parquet8")

combined_df=pl.concat([df1, df2, df3, df4, df5, df6], how="vertical")
print(combined_df)

shape: (19_166_425, 4)
┌──────────────────────────┬─────────────────────────┬─────────────────────────┬───────────────────┐
│ inserted_at              ┆ issued                  ┆ delivery_start          ┆ residual_load_fcs │
│ ---                      ┆ ---                     ┆ ---                     ┆ ---               │
│ datetime[μs,             ┆ datetime[μs,            ┆ datetime[μs,            ┆ f64               │
│ Europe/Berlin]           ┆ Europe/Berlin]          ┆ Europe/Berlin]          ┆                   │
╞══════════════════════════╪═════════════════════════╪═════════════════════════╪═══════════════════╡
│ 2026-02-05               ┆ 2026-02-05 17:00:00 CET ┆ 2026-01-29 18:00:00 CET ┆ 51058.537106      │
│ 17:47:42.928874 CET      ┆                         ┆                         ┆                   │
│ 2026-02-05               ┆ 2026-02-05 17:00:00 CET ┆ 2026-01-29 18:15:00 CET ┆ 50818.339595      │
│ 17:47:42.928874 CET      ┆                         ┆              

In [2]:
energy_weather = (
    combined_df
    .sort("issued")
    .group_by("delivery_start", maintain_order=True)
    .agg(
        pl.col("issued").last().alias("latest_issued"),
        pl.col("inserted_at").last(),
        pl.col("residual_load_fcs").last(),
    )
)

print(energy_weather)

shape: (17_980, 4)
┌──────────────────────────┬─────────────────────────┬─────────────────────────┬───────────────────┐
│ delivery_start           ┆ latest_issued           ┆ inserted_at             ┆ residual_load_fcs │
│ ---                      ┆ ---                     ┆ ---                     ┆ ---               │
│ datetime[μs,             ┆ datetime[μs,            ┆ datetime[μs,            ┆ f64               │
│ Europe/Berlin]           ┆ Europe/Berlin]          ┆ Europe/Berlin]          ┆                   │
╞══════════════════════════╪═════════════════════════╪═════════════════════════╪═══════════════════╡
│ 2026-01-29 18:00:00 CET  ┆ 2026-02-05 17:00:00 CET ┆ 2026-02-05              ┆ 51058.537106      │
│                          ┆                         ┆ 17:47:42.928874 CET     ┆                   │
│ 2026-01-29 18:15:00 CET  ┆ 2026-02-05 18:00:00 CET ┆ 2026-02-05              ┆ 50818.339595      │
│                          ┆                         ┆ 18:02:41.876738 C

In [3]:
import polars as pl
#insert cutoff column to filter out rows where issued is greater than cutoff: 12:00 PM of the previous day of delivery_start
df = combined_df.with_columns(
    (
        pl.col("delivery_start").dt.truncate("1d")
        - pl.duration(days=1)
        + pl.duration(hours=12)
    ).alias("cutoff")
)

energy_weather = (
    df
    .filter(pl.col("issued") <= pl.col("cutoff"))
    .sort("issued")
    .group_by("delivery_start")
    .last()
    .sort("delivery_start")
)
print(energy_weather)

shape: (17_188, 5)
┌───────────────────┬───────────────────┬───────────────────┬───────────────────┬──────────────────┐
│ delivery_start    ┆ inserted_at       ┆ issued            ┆ residual_load_fcs ┆ cutoff           │
│ ---               ┆ ---               ┆ ---               ┆ ---               ┆ ---              │
│ datetime[μs,      ┆ datetime[μs,      ┆ datetime[μs,      ┆ f64               ┆ datetime[μs,     │
│ Europe/Berlin]    ┆ Europe/Berlin]    ┆ Europe/Berlin]    ┆                   ┆ Europe/Berlin]   │
╞═══════════════════╪═══════════════════╪═══════════════════╪═══════════════════╪══════════════════╡
│ 2026-02-07        ┆ 2026-02-06        ┆ 2026-02-06        ┆ 39307.856365      ┆ 2026-02-06       │
│ 00:00:00 CET      ┆ 12:47:40.573730   ┆ 12:00:00 CET      ┆                   ┆ 12:00:00 CET     │
│                   ┆ CET               ┆                   ┆                   ┆                  │
│ 2026-02-07        ┆ 2026-02-06        ┆ 2026-02-06        ┆ 38762.3725

In [4]:
energy_weather.write_parquet("energy_weather_data.parquet")
